# Lab 1 Part 2 — Tool Calling, ReAct, and SQL Agents

**Day 1 Morning Extension | ~45 minutes | CPU | OpenAI API key required**

Part 1 showed the LLM as an inference engine: you send text, it returns text. Part 2 changes the shape of the system. Now the LLM can ask your program to run a tool. Your program performs the action, sends the result back, and the LLM uses that observation to answer.

This is the foundation underneath practical agents. We will build the idea in four steps:

1. **Tool calling:** the model requests a structured function call instead of guessing.
2. **Real API tool:** the model calls a no-key weather API through your code.
3. **ReAct:** the model alternates between reasoning and acting.
4. **SQL agent:** the model turns a natural-language question into a safe database query, your code runs it, and the model explains the result.

> The big idea: the model does not directly touch your database or APIs. You expose carefully described tools. The model asks to use them. Your code decides what actually runs.

In [ ]:
!uv pip install -q openai pandas httpx
print('Install complete')


In [ ]:
# Configuration — works in Colab Secrets or local environment variables
import os

def get_secret(name: str, *, required: bool = True):
    value = os.environ.get(name)
    try:
        from google.colab import userdata
        value = userdata.get(name) or value
    except Exception:
        pass
    if required and not value:
        raise ValueError(
            f"Missing {name}. In Colab, open the key icon in the left sidebar, "
            f"add a secret named {name}, paste the value, and enable Notebook access."
        )
    return value

OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
DEFAULT_MODEL = 'gpt-4o-mini'

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

print(f'Config loaded — OPENAI_API_KEY starts with {OPENAI_API_KEY[:8]}...')


## Part A — From Text Generation to Tool Use

A normal chat completion asks the model to answer directly. Tool calling gives the model another option: it can return a structured request such as:

```json
{
  "name": "estimate_model_memory",
  "arguments": {"params_b": 7, "precision": "int4"}
}
```

That request is not executed by the model. It is executed by your Python code. This boundary matters:

- The **model decides** which tool it wants and what arguments to pass.
- Your **application validates and executes** the tool.
- The **model observes** the tool result and writes the final answer.

This is why tool/function descriptions are part of your product surface. If the tool schema is vague, the model will use it badly.


In [ ]:
# A tiny tool: estimate model memory from parameter count and precision.
# This is deliberately simple so we can see the tool-calling mechanics clearly.
import json

def estimate_model_memory(params_b: float, precision: str) -> str:
    bytes_per_param = {
        'fp32': 4.0,
        'fp16': 2.0,
        'bf16': 2.0,
        'int8': 1.0,
        'int4': 0.5,
        'nf4': 0.5,
    }
    key = precision.lower()
    if key not in bytes_per_param:
        return json.dumps({'error': f'Unsupported precision: {precision}'})
    memory_gb = params_b * 1e9 * bytes_per_param[key] / 1e9
    return json.dumps({
        'params_b': params_b,
        'precision': key,
        'estimated_weight_memory_gb': round(memory_gb, 2),
    })

print(estimate_model_memory(7, 'int4'))


### Describe the Tool to the Model

The model cannot inspect your Python function. It only sees the schema you send in the API call. The schema tells it:

- the function name,
- what the function does,
- what arguments it accepts,
- which arguments are required.

This is a contract. The better the contract, the better the tool use.


In [ ]:
memory_tool = {
    'type': 'function',
    'function': {
        'name': 'estimate_model_memory',
        'description': (
            'Estimate the weight memory in GB for an LLM from parameter count and precision. '
            'Use this for deployment sizing questions.'
        ),
        'parameters': {
            'type': 'object',
            'properties': {
                'params_b': {
                    'type': 'number',
                    'description': 'Model size in billions of parameters, such as 7 or 13.'
                },
                'precision': {
                    'type': 'string',
                    'enum': ['fp32', 'fp16', 'bf16', 'fp8', 'int8', 'int4', 'nf4'],
                    'description': 'The numeric precision used to store model weights.'
                },
            },
            'required': ['params_b', 'precision'],
        },
    },
}


### The Two-Round Tool Calling Loop

Before wrapping anything in a helper function, we will inspect the mechanics one cell at a time. This matters because `finish_reason` changes meaning:

- Without tools, the model usually stops with `finish_reason='stop'` and returns text.
- With tools, the model may stop with `finish_reason='tool_calls'` and return no final answer yet.

That second case means: **the model is asking your application to act.**


In [ ]:
# TC-1 — Normal response without tools: the model answers directly.
plain_messages = [
    {'role': 'user', 'content': 'How much memory does a 7B INT4 model need?'}
]
plain = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=plain_messages,
)
print('finish_reason:', plain.choices[0].finish_reason)
print('content:')
print(plain.choices[0].message.content)


In [ ]:
# print with formatting
import re
from IPython.display import display, Markdown

def fix_latex(text):
    # \[...\] → $$...$$  (block math)
    text = re.sub(r'\\\[(.*?)\\\]', r'$$\1$$', text, flags=re.DOTALL)
    # \(...\) → $...$  (inline math)
    text = re.sub(r'\\\((.*?)\\\)', r'$\1$', text, flags=re.DOTALL)
    return text

content = plain.choices[0].message.content
display(Markdown(fix_latex(content)))

In the normal response, the model gives an answer immediately. It may calculate correctly, but it is still generating from its learned patterns. There is no external computation and no explicit tool result.

In the next cell we use `tool_choice={'type': 'function', 'function': {'name': 'estimate_model_memory'}}` to **force** a tool call. This guarantees a `tool_calls` response for the demonstration. In TC-4 and production code, use `tool_choice='auto'` and let the model decide whether a tool is needed.

In [ ]:
# TC-2 — Add a tool: the model can ask your code to run a function.
tool_messages = [
    {'role': 'user', 'content': 'How much memory does a 7B INT4 model need?'}
]
tool_response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=tool_messages,
    tools=[memory_tool],
    tool_choice={'type': 'function', 'function': {'name': 'estimate_model_memory'}},
)

choice = tool_response.choices[0]
print('finish_reason:', choice.finish_reason)
print('content:', choice.message.content)
print('tool_calls:')
print(choice.message.tool_calls)


When `finish_reason` is `tool_calls`, the model has not finished answering. It has paused and returned one or more structured action requests. Your application now owns the next step.

Important API rule: if the assistant message contains multiple `tool_calls`, your message history must include one `role='tool'` response for **every** `tool_call_id` before you ask the model to continue. Comparison questions often trigger parallel tool calls, such as one call for INT4 and another call for FP16.

In [ ]:
# TC-3 — Manual Round 2: run every requested tool and return every observation.
assistant_message = tool_response.choices[0].message
round2_messages = [
    {'role': 'user', 'content': 'How much memory does a 7B INT4 model need?'},
    assistant_message,
]

for tool_call in assistant_message.tool_calls:
    args = json.loads(tool_call.function.arguments)
    print('Model wants to call:', tool_call.function.name)
    print('Arguments:', args)

    if tool_call.function.name != 'estimate_model_memory':
        tool_result = json.dumps({'error': f'Unknown tool: {tool_call.function.name}'})
    else:
        tool_result = estimate_model_memory(**args)

    print('Tool result:', tool_result)
    round2_messages.append({
        'role': 'tool',
        'tool_call_id': tool_call.id,
        'content': tool_result,
    })

final = client.chat.completions.create(model=DEFAULT_MODEL, messages=round2_messages)
print('finish_reason:', final.choices[0].finish_reason)
print('Final answer:')
print(final.choices[0].message.content)

This prints the full conversation the model sees before writing its final answer. Notice the three roles: `user`, `assistant` with `tool_calls`, and `tool` with the result. This is the protocol. Every tool-calling framework — LangChain, LlamaIndex, AutoGen — assembles exactly this structure under the hood.

In [ ]:
import pprint
print("=== Full message history after Round 2 ===")
pprint.pprint(round2_messages)

Now the two-round loop should be visible. The helper function below does the same thing, but now you know exactly what it automates.

Notice that this is still a **one-tool agent** in the sense that it exposes one Python function, `estimate_model_memory`. But the model may call that same tool multiple times in one assistant turn. The wrapper must therefore loop over all tool calls before making Round 2.

In [ ]:
# TC-4 — Wrap the mechanics in a reusable one-tool agent.
def memory_agent(question: str) -> str:
    messages = [
        {'role': 'system', 'content': 'You answer LLM deployment sizing questions. Use tools when helpful.'},
        {'role': 'user', 'content': question},
    ]

    first = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=messages,
        tools=[memory_tool],
        tool_choice='auto',
    )
    first_message = first.choices[0].message

    if not first_message.tool_calls:
        return first_message.content

    messages.append(first_message)

    for tool_call in first_message.tool_calls:
        args = json.loads(tool_call.function.arguments)
        print(f"Tool requested: {tool_call.function.name}({args})")

        if tool_call.function.name != 'estimate_model_memory':
            tool_result = json.dumps({'error': f'Unknown tool: {tool_call.function.name}'})
        else:
            tool_result = estimate_model_memory(**args)

        messages.append({
            'role': 'tool',
            'tool_call_id': tool_call.id,
            'content': tool_result,
        })

    second = client.chat.completions.create(model=DEFAULT_MODEL, messages=messages)
    return second.choices[0].message.content

print(memory_agent('How much weight memory does a 7B model need in INT4 versus FP16?'))

Tools should **return structured errors**, not crash the agent loop. In this small classroom tool, `fp8` is allowed through the schema as a user-facing precision term, but the Python implementation does not support it yet. Watch the tool return JSON like `{"error": "Unsupported precision: fp8"}`, and the model handle that observation in its response. If the tool raised an exception instead, the notebook would stop before the model could recover.

In [ ]:
# What happens when the model passes an unsupported precision?
print(memory_agent("How much memory does a 7B model need in fp8?"))

## Part B — A Real External API Tool: Weather

Tools are not limited to local Python math. They can call web APIs, internal services, filesystems, databases, or deployment platforms. This example uses Open-Meteo, a free weather API that does not require an API key.

The interesting part is that the user can ask for weather in a city, while the tool requires latitude and longitude. The model uses its general knowledge to choose coordinates; the tool provides current external data.

> **Note on External APIs:** The tool below uses `open-meteo.com`. If your environment blocks external HTTP calls or the API is unreachable, comment out the `httpx` call and return a mock JSON string instead so you can continue the lab.


In [ ]:
import httpx

def get_current_weather(latitude: float, longitude: float) -> str:
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'current_weather': True,
    }
    response = httpx.get(url, params=params, timeout=10)
    response.raise_for_status()
    return response.text

weather_tool = {
    'type': 'function',
    'function': {
        'name': 'get_current_weather',
        'description': 'Get current weather for a latitude and longitude using the Open-Meteo API.',
        'parameters': {
            'type': 'object',
            'properties': {
                'latitude': {'type': 'number', 'description': 'Latitude of the location.'},
                'longitude': {'type': 'number', 'description': 'Longitude of the location.'},
            },
            'required': ['latitude', 'longitude'],
        },
    },
}


In [ ]:
weather_messages = [
    {'role': 'user', 'content': 'What is the current weather in Amman, Jordan?'}
]
weather_response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=weather_messages,
    tools=[weather_tool],
    tool_choice={'type': 'function', 'function': {'name': 'get_current_weather'}},
)

choice = weather_response.choices[0]
print('finish_reason:', choice.finish_reason)

final_weather_messages = [weather_messages[0], choice.message]
for call in choice.message.tool_calls:
    args = json.loads(call.function.arguments)
    print('Tool requested:', call.function.name)
    print('Arguments:', args)

    if call.function.name != 'get_current_weather':
        weather_result = json.dumps({'error': f'Unknown tool: {call.function.name}'})
    else:
        weather_result = get_current_weather(**args)

    print('Weather API result:', weather_result[:300])
    final_weather_messages.append({
        'role': 'tool',
        'tool_call_id': call.id,
        'content': weather_result,
    })

final_weather = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=final_weather_messages,
)
print('Final answer:')
print(final_weather.choices[0].message.content)

In [ ]:
# display markdown
from IPython.display import display, Markdown
display(Markdown(final_weather.choices[0].message.content))

## Part C — ReAct: Reasoning + Acting

ReAct means **Reason + Act**. The model does not only produce an answer. It works in a loop:

```text
Thought: What do I need to know?
Action: Call a tool.
Observation: Read the tool result.
Thought: Do I have enough information?
Answer: Respond to the user.
```

You already used this pattern in Parts A and B. The model chose an action, your code produced an observation, and the model used that observation to answer.

Modern tool calling returns the **Action** as structured JSON. ReAct was originally often implemented by making the model print structured text that your code parsed with regex. Here's what that looks like — run it and watch the Thought/Action/Observation loop print in real time.

In [ ]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
            model=DEFAULT_MODEL,
            temperature=0,
            messages=self.messages,
        )
        return completion.choices[0].message.content

⚠️ **Security Warning for `eval()`:** The `calculate` function below uses `eval()` to execute arbitrary Python code. It is acceptable for this classroom toy because the tool only processes generated arithmetic strings. **Never use `eval()` on untrusted user input in production.**


In [ ]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier":
        return "Scottish Terriers average 20 lbs"
    elif name in "Border Collie":
        return "a Border Collies average weight is 37 lbs"
    elif name in "Toy Poodle":
        return "a toy poodles average weight is 7 lbs"
    else:
        return "An average dog weights 50 lbs"

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight,
}

In [ ]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer.
Use Thought to briefly describe what you need to do.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look up the dog's weight using average_dog_weight.
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weighs 51 lbs

You then output:

Answer: A Bulldog weighs 51 lbs
""".strip()

In [ ]:
import re

action_re = re.compile(r'^Action: (\w+): (.*)$')

def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(line)
            for line in result.splitlines()
            if action_re.match(line)
        ]
        if actions:
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise ValueError(f"Unknown action: {action}: {action_input}")
            print(f" -- running {action} {action_input}")
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = f"Observation: {observation}"
        else:
            return

In [ ]:
query("I have 2 dogs, a border collie and a scottish terrier. What is their combined weight?")

### Text-Based vs. Structured ReAct

| | Text-Based (above) | Structured (what you built) |
|---|---|---|
| **Action format** | Plain text: `Action: tool_name: args` | JSON: `{"name": "...", "arguments": {...}}` |
| **Parsing** | Regex — breaks if model adds punctuation | API-level — valid structured arguments |
| **Reliability** | Fragile — format must be exact | Production-grade |
| **When you'll see it** | Legacy code, research papers | Modern APIs (OpenAI, Anthropic, Gemini) |

The idea is identical. The transport is different. Everything you built in Parts A and B **is** ReAct — just with a reliable format.

## Part D — SQL Agent From Scratch

Now we build the practical version. The user asks a natural-language question about model benchmarks. The LLM does not know the table contents. Instead, it gets a `run_sql` tool and a schema description.

The agent loop is still the same pattern you have already seen:

1. User asks a question.
2. Model decides whether a tool is needed.
3. Your Python code validates and runs the tool.
4. Model reads the observation and writes the final answer.

The difference is the tool now reaches into a database, so the safety boundary matters much more.

In [ ]:
# Build a small file-backed benchmark database.
# SQLite is built into Python, and the .db file is visible in Colab's file browser.
import os
import sqlite3
import pandas as pd

DB_PATH = 'benchmarks.db'
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)  # fresh start each run

conn = sqlite3.connect(DB_PATH)
conn.execute('''CREATE TABLE models (
    name TEXT, params_b REAL, vram_gb REAL,
    tokens_per_sec INTEGER, quality REAL
)''')
conn.executemany('INSERT INTO models VALUES (?,?,?,?,?)', [
    ('Qwen2.5-0.5B', 0.5, 2.0, 120, 6.2),
    ('Qwen2.5-1.5B', 1.5, 4.0,  85, 7.1),
    ('Qwen2.5-7B',   7.0,14.0,  40, 8.3),
    ('Llama-3.2-3B', 3.0, 6.0,  65, 7.4),
    ('Llama-3.1-8B', 8.0,16.0,  35, 8.5),
    ('Mistral-7B',   7.0,14.0,  42, 8.1),
])
conn.commit()

pd.read_sql_query('SELECT * FROM models', conn)

`benchmarks.db` is now visible in the Colab file browser (left sidebar → folder icon). You can download it and open it with [DB Browser for SQLite](https://sqlitebrowser.org/) to inspect the data the agent will query.

### Build a Safe SQL Tool

A real SQL agent must be constrained. Never hand an LLM arbitrary write access to a production database. In this classroom version, we enforce the minimum safety rules:

- only `SELECT` queries,
- one statement at a time,
- no comments or semicolon chaining,
- only the `models` table,
- row limit applied by the application,
- error messages returned as data instead of crashing the notebook.

There are two functions in the next cell:

- `is_safe_select()` is the **gatekeeper**. It does not run SQL. It only checks whether the model-generated query is allowed.
- `run_sql()` is the **tool implementation**. It calls the validator, opens `benchmarks.db`, executes approved queries, fetches at most 20 rows, and returns JSON.

The LLM never gets a raw database connection. It only gets access to this constrained Python function. That separation is the safety boundary.

In [ ]:
import re

def is_safe_select(query: str) -> tuple[bool, str]:
    normalized = ' '.join(query.strip().split()).lower()
    if not normalized.startswith('select '):
        return False, 'Only SELECT queries are allowed.'
    blocked = [';', '--', '/*', '*/', ' pragma ', ' attach ', ' detach ', ' drop ', ' delete ', ' update ', ' insert ', ' alter ', ' create ']
    if any(token in f' {normalized} ' for token in blocked):
        return False, 'Query contains a blocked token or multiple-statement pattern.'

    table_refs = re.findall(r'\b(?:from|join)\s+([a-zA-Z_][\w]*)', normalized)
    if not table_refs:
        return False, 'Query must read from the models table.'
    if any(table != 'models' for table in table_refs):
        return False, 'Query may only read from the models table.'
    return True, 'ok'

def run_sql(query: str) -> str:
    ok, reason = is_safe_select(query)
    if not ok:
        return json.dumps({'error': reason, 'query': query})
    try:
        with sqlite3.connect(DB_PATH) as con:
            cur = con.execute(query)
            rows = cur.fetchmany(20)
            cols = [d[0] for d in cur.description]
            return json.dumps([dict(zip(cols, row)) for row in rows])
    except Exception as e:
        return json.dumps({'error': str(e), 'query': query})

print(run_sql('SELECT name, vram_gb FROM models WHERE vram_gb <= 8 ORDER BY vram_gb'))
print(run_sql('SELECT name FROM models_v2'))

The first call returns real rows — the query is valid and safe.

The second call returns an error — this is **intentional**. The validator correctly rejects `models_v2` because only the `models` table is allowed. This is the safety guardrail working as designed.

### Describe the Database Tool

The tool description gives the model enough schema context to write useful SQL. Notice that we include table and column names in the description. Without that, the model has to guess.


In [ ]:
sql_tool = {
    'type': 'function',
    'function': {
        'name': 'run_sql',
        'description': (
            'Run a read-only SQLite SELECT query against the models table. '
            'The table columns are: name TEXT, params_b REAL, vram_gb REAL, '
            'tokens_per_sec INTEGER, quality REAL from 0 to 10. '
            'Only use SELECT queries. Do not use semicolons.'
        ),
        'parameters': {
            'type': 'object',
            'properties': {
                'query': {
                    'type': 'string',
                    'description': 'A single read-only SQLite SELECT query over the models table.'
                }
            },
            'required': ['query'],
        },
    },
}


### The SQL Agent Loop

Read this slowly. There is no framework here:

1. We send the user question plus the SQL tool schema.
2. The model requests a SQL query.
3. We validate and execute the query.
4. We send rows back as the tool observation.
5. The model writes the final answer.

That is the heart of a SQL agent. It is ReAct over a database: reason about the question, act by querying, observe rows, answer.

In [ ]:
def sql_agent(question: str) -> str:
    messages = [
        {
            'role': 'system',
            'content': (
                'You answer questions about LLM benchmark data. '
                'Use the run_sql tool when the answer requires table data. '
                'When you answer, mention the evidence from the SQL result.'
            ),
        },
        {'role': 'user', 'content': question},
    ]

    # Round 1: model decides what query would answer the question.
    first = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=messages,
        tools=[sql_tool],
        tool_choice='auto',
        temperature=0,
    )
    first_message = first.choices[0].message

    if not first_message.tool_calls:
        return first_message.content

    messages.append(first_message)

    for call in first_message.tool_calls:
        args = json.loads(call.function.arguments)

        if call.function.name != 'run_sql':
            print(f'Unknown tool requested: {call.function.name}')
            result = json.dumps({'error': f'Unknown tool: {call.function.name}'})
        else:
            query = args['query']
            print(f'SQL requested: {query}')
            result = run_sql(query)

        print(f'Tool observation: {result[:180]}...')
        messages.append({
            'role': 'tool',
            'tool_call_id': call.id,
            'content': result,
        })

    # Round 2: model reads the observation and answers in natural language.
    second = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=messages,
        temperature=0.2,
    )
    return second.choices[0].message.content

Before connecting the agent to data, see what the model says on its own. It will produce a plausible-sounding answer, but it has no idea what's in your `benchmarks.db`. This is why tools exist.

In [ ]:
# Before the agent: ask the same question without the tool.
baseline = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{'role': 'user', 'content': 'Which models fit in 8 GB of VRAM?'}],
    temperature=0,
)
print("=== Without tool (model guesses from training data) ===")
print(baseline.choices[0].message.content)
print()
print("=== With SQL agent (model queries real data) ===")
print(sql_agent('Which models fit in 8 GB of VRAM?'))

In [ ]:
questions = [
    'Which models fit in 8 GB of VRAM?',
    'What is the fastest model with quality above 8.0?',
    'What is the average quality of models under 5B parameters?',
]

for q in questions:
    print(f'Q: {q}')
    print(sql_agent(q))
    print('-' * 80)


## Part E — What You Just Built

You built a small but real agentic system:

- **Tool calling:** the model selected a function and supplied structured arguments.
- **External API tool:** the model requested weather data through your code.
- **ReAct loop:** the model reasoned about needing data, acted through a tool, observed the result, and answered.
- **SQL agent:** natural-language questions became safe SQL queries over a database.

The model did not magically become connected to a database. You connected it by exposing controlled tools. That is the deployment pattern.

## Production Guardrails

Before pointing this pattern at real data, add stronger controls:

- Use a read-only database user.
- Keep an allowlist of tables and columns.
- Parse SQL with a SQL parser rather than string checks.
- Add row limits and timeouts.
- Log every generated query.
- Add human review for sensitive domains.
- Trace tool calls with observability tools.

## Lab 1 Part 2 Complete

You should now have:

- [ ] A working one-tool memory sizing agent.
- [ ] A working external API tool call.
- [ ] A clear mental model for ReAct.
- [ ] A working SQL agent from scratch.
- [ ] A list of safety controls required for production SQL agents.

## Exercise

Add a new tool called `get_efficiency_score` that takes a model `name` string, queries `benchmarks.db` for that model's `quality` and `tokens_per_sec`, and returns `round(quality / tokens_per_sec * 100, 2)` as a JSON string.

Then ask the SQL agent: **Which model has the best quality-per-speed efficiency?**

You will need to:

1. Write the Python function.
2. Write the tool schema.
3. Add it to the agent's `tools` list.
4. Update the agent loop so it can dispatch either `run_sql` or `get_efficiency_score`.

## Stretch Goals

1. Add a `cost_per_1k_tokens` column and ask for the cheapest high-quality model.
2. Add a second tool called `describe_schema()` and let the model inspect the schema before querying.
3. Add a stricter validator that rejects `SELECT *`.
4. Compare this scratch implementation with a LangChain SQL agent.
5. **Parallel tool calls** — Ask `memory_agent('Compare memory for 7B in INT4, FP16, and BF16')`. After the call, inspect `first_message.tool_calls` — how many items are in the list? Notice all three calls happen in a single assistant turn. This is parallel tool execution and it is one of the key efficiency features of the structured tool-calling API.